## Rendering detours

Reranking left two encoded polylines on every result. This notebook draws them with plain
[`ipyleaflet`](https://ipyleaflet.readthedocs.io) objects — `Map`, `Polyline`, `Marker`,
`GeoJSON` — instead of the demo `ResponseMap`/`RouteController` pair, to show how little is
needed to plug the feature into an existing map.

The cell below repeats the route, the search and the reranking of the
[previous notebook](obm_12_detour_api.ipynb):

In [ ]:
import asyncio

from flexpolyline import decode, encode

from here_search_demo.api import API
from here_search_demo.auth import Credentials
from here_search_demo.http import HTTPSession
from here_search_demo.routing_api import RoutingAPI, Waypoint
from here_search_demo.widgets.route_geometry import simplify_polyline

credentials = Credentials()
api = API(credentials=credentials)
routing_api = RoutingAPI(credentials=credentials)
session = HTTPSession()

start_pos =  48.57436, 7.71556  # Strasbourg
stop_pos =  45.75049, 4.84418   # Lyon
at_pos =  48.03529, 7.37121     # somewhere on the route

route = await routing_api.routes(session=session, origin=start_pos, destination=stop_pos)
(route_section,) = RoutingAPI.sections(route)

# The detour baseline is the *remaining* journey, at → stop, not the whole route.
remaining = await routing_api.routes(session=session, origin=at_pos, destination=stop_pos)
(baseline,) = RoutingAPI.sections(remaining)

resp = await api.browse(
    session=session,
    latitude=at_pos[0],
    longitude=at_pos[1],
    categories=["100-1000", ],
    polyline=encode(simplify_polyline(decode(route_section.polyline), 2000)),
    width=500,
    limit=20,
    lang="en",
)


async def detour_legs(item):
    position = item["position"]
    response = await routing_api.routes(
        session=session,
        origin=at_pos,
        destination=stop_pos,
        via=Waypoint(position["lat"], position["lng"], item.get("address", {}).get("street")),
        return_fields=("summary", "polyline"),
    )
    return RoutingAPI.sections(response)


items = [item for item in resp.data["items"] if "position" in item]
legs = await asyncio.gather(*(detour_legs(item) for item in items))

detours = []
for item, (to_leg, from_leg) in zip(items, legs):
    detours.append(
        {
            "title": item["title"],
            "position": (item["position"]["lat"], item["position"]["lng"]),
            "dur_to_sec": to_leg.duration,
            "dur_from_sec": from_leg.duration,
            "excursion_m": to_leg.length + from_leg.length - baseline.length,
            "polyline_to": to_leg.polyline,
            "polyline_from": from_leg.polyline,
        }
    )
detours.sort(key=lambda detour: detour["dur_to_sec"])

[detour["title"] for detour in detours]

### The map and the route

`ipyleaflet` defaults to an OpenStreetMap basemap. Pointing a `TileLayer` at the
[HERE Map Tile API v3][1] instead only takes the API key already held by `Credentials`;
this is exactly what `widgets.input_map.PositionMap` does, with `lite.day` as the default
style and `satellite.day` as the alternative.

The route polyline then decodes to a list of `(latitude, longitude)` pairs, which is what
an `ipyleaflet.Polyline` expects as `locations`:

[1]: https://www.here.com/docs/bundle/map-tile-api-v3-api-reference/page/index.html

In [ ]:
from ipyleaflet import Map, Marker, Polyline, TileLayer, AwesomeIcon

here_tiles = TileLayer(
    url=(
        "https://maps.hereapi.com/v3/base/mc/{z}/{x}/{y}/png"
        f"?style=lite.day&ppi=400&size=512&apiKey={credentials.api_key}&lang=en"
    ),
    attribution="© HERE 2026",
)

route_map = Map(
    center=((start_pos[0] + stop_pos[0]) / 2, (start_pos[1] + stop_pos[1]) / 2) , 
    zoom=6, basemap={}, 
    layout={"height": "500px"},
    zoom_control=False,
    scroll_wheel_zoom=True,
    close_popup_on_click=False,
    double_click_zoom=False,
    box_zoom=True)
route_map.add(here_tiles)

route_line = Polyline(locations=decode(route_section.polyline), color="#2c7fb8", weight=2, fill=False)
route_map.add(route_line)

route_map.add(Marker(location=start_pos, title="start", draggable=False, icon=AwesomeIcon(name="play")))
route_map.add(Marker(location=at_pos, title="at", draggable=False, icon=AwesomeIcon(name="car")))
route_map.add(Marker(location=stop_pos, title="stop", draggable=False, icon=AwesomeIcon(name="flag")))

display(route_map)

### The reranked results

The results are added as a single `GeoJSON` layer rather than as one marker each: one
layer redraws faster, and its `on_click` callback carries the feature properties, which is
where the detour annotation is kept.

In [ ]:
from ipyleaflet import GeoJSON

collection = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [detour["position"][1], detour["position"][0]]},
            "properties": {
                "rank": rank,
                "title": detour["title"],
                "label": f"{detour['title']} — {round(detour['dur_to_sec'] / 60)} min, "
                f"{detour['excursion_m'] / 1000:+.1f} km",
            },
        }
        for rank, detour in enumerate(detours)
    ],
}

results_layer = GeoJSON(
    data=collection,
    point_style={"radius": 7, "color": "#2c7fb8", "fillOpacity": 0.8, "weight": 1},
    hover_style={"fillColor": "#f57c00"},
)
route_map.add(results_layer)

[feature["properties"]["label"] for feature in collection["features"]]

### Drawing one detour

Selecting a result draws its two legs: the `at → result` leg in **green**, the
`result → stop` leg in **purple**. Both polylines were returned by the reranking call, so
this costs no Routing API request at all, so hovering or arrow-keying through the
result list stays responsive.

In [ ]:
detour_layers = []


def draw_detour(rank):
    """Draw the two legs of the detour of the result at *rank*."""
    clear_detour()
    detour = detours[rank]
    for polyline, color in ((detour["polyline_to"], "#2ca25f"), (detour["polyline_from"], "#8856a7")):
        layer = Polyline(locations=decode(polyline), color=color, weight=2, fill=False)
        detour_layers.append(layer)
        route_map.add(layer)
    return detour


def clear_detour():
    while detour_layers:
        route_map.remove(detour_layers.pop())


def on_result_click(event=None, feature=None, properties=None, **kwargs):
    draw_detour(properties["rank"])


results_layer.on_click(on_result_click)

drawn = draw_detour(0)
len(detour_layers), drawn["title"]

Another result means a `clear_detour()` / `draw_detour()` pair; leaving the result list
means a `clear_detour()`:

In [ ]:
draw_detour(len(detours) - 1)
clear_detour()
len(detour_layers)

### In the demo application

`widgets.input_map.PositionMap` builds the same HERE tile layer (and falls back to a
Positron basemap when no API key is available).
`widgets.route.RouteController.draw_detour_routes()` and `clear_detour_routes()` are the
same two functions, applied to a `ResponseMap`; `widgets.output_map.ResponseMap.click_result()`
is the `on_click` handler, and it reads the polylines from the route cache instead of a
local list. The [`ResponseMap` variant of this notebook](obm_8_detour_map.ipynb) shows that
path, and the [app automation notebook](../obm_5_automate.ipynb) drives the complete
application.

<!--
from urllib.parse import quote, unquote
svg = """<svg width="40" height="48" viewBox="0 0 40 48" fill="none" xmlns="http://www.w3.org/2000/svg"><path class="triangle" d="M11,36.8l-5.5,5.5L0,36.8H11z" fill="#48DAD0"></path><path class="HERE" d="M19.1,24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9,1.7-0.5,2.7,0.5L19.1,24.2z M30.9,9.2c0.9-0.9,1.7-0.5,2.7,0.5
    L31.3,12C30.1,10.7,30.2,9.9,30.9,9.2z M38,11.6c-1.1,1.6-2.9,4.2-4.9,2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4
	c-1.6,1.6-2,3.4-1.5,5.1l-1.6-1.8c-0.5,0.3-2.4,1.9-0.9,4.5l-1.8-1.5l-2.4,2.4l3.2,3.2c-2.5-1.9-5.1-1.8-7.2,0.4
	c-2.3,2.3-2.1,5-0.4,7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9,0.9-1.5,2.2-1.3,3.1L4,24.6l-2.6,2.6l9.6,9.6h5.2l-3.5-3.5
	c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8,1.8-0.3,3.6,1.4l3.4,3.4l2.6-2.6l-3.2-3.2c2.3,1.8,5.1,1.9,7.7-0.6l0,0c1.5-1.4,2-2.8,2-2.8
	l-1.9-1.3c-1.1,1.6-2.9,4.2-4.9,2.3l5-5l3.1,3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5,0-4.1c0.4,0.7,0.9,1.5,1.5,2.1
	c2.5,2.5,5.7,3,8.6,0.2l0,0c1.5-1.4,2-2.8,2-2.8S38,11.6,38,11.6z" fill="#000000"></path></svg>
"""
print(f"""![HERE](data:image/svg+xml,{quote(svg)})
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>""")
-->
![HERE](data:image/svg+xml,%3Csvg%20width%3D%2240%22%20height%3D%2248%22%20viewBox%3D%220%200%2040%2048%22%20fill%3D%22none%22%20xmlns%3D%22http%3A//www.w3.org/2000/svg%22%3E%3Cpath%20class%3D%22triangle%22%20d%3D%22M11%2C36.8l-5.5%2C5.5L0%2C36.8H11z%22%20fill%3D%22%2348DAD0%22%3E%3C/path%3E%3Cpath%20class%3D%22HERE%22%20d%3D%22M19.1%2C24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5L19.1%2C24.2z%20M30.9%2C9.2c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5%0A%20%20%20%20L31.3%2C12C30.1%2C10.7%2C30.2%2C9.9%2C30.9%2C9.2z%20M38%2C11.6c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4%0A%09c-1.6%2C1.6-2%2C3.4-1.5%2C5.1l-1.6-1.8c-0.5%2C0.3-2.4%2C1.9-0.9%2C4.5l-1.8-1.5l-2.4%2C2.4l3.2%2C3.2c-2.5-1.9-5.1-1.8-7.2%2C0.4%0A%09c-2.3%2C2.3-2.1%2C5-0.4%2C7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9%2C0.9-1.5%2C2.2-1.3%2C3.1L4%2C24.6l-2.6%2C2.6l9.6%2C9.6h5.2l-3.5-3.5%0A%09c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8%2C1.8-0.3%2C3.6%2C1.4l3.4%2C3.4l2.6-2.6l-3.2-3.2c2.3%2C1.8%2C5.1%2C1.9%2C7.7-0.6l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8%0A%09l-1.9-1.3c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.3l5-5l3.1%2C3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5%2C0-4.1c0.4%2C0.7%2C0.9%2C1.5%2C1.5%2C2.1%0A%09c2.5%2C2.5%2C5.7%2C3%2C8.6%2C0.2l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8S38%2C11.6%2C38%2C11.6z%22%20fill%3D%22%23000000%22%3E%3C/path%3E%3C/svg%3E%0A)
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>